In [1]:
# notebooks live in notebooks/, the project modules in the repo root
import sys
if '..' not in sys.path:
    sys.path.insert(0, '..')

%load_ext autoreload
%autoreload 2

# standard libraries
import math

# third-party
import numpy as np
import pandas as pd
import torch
import seaborn as sns
from matplotlib import pyplot as plt

# project modules
import benchmarks as bm
from model import Policy
from market.simulators import simulate_heston, simulate_gbm, HestonParams
from market.pricing import bs_price, bs_delta, bs_gamma, heston_price
from features import nn_policy, ntb_policy
from train import train
from objectives import EntropicLoss, CVaRLoss


In [2]:
# shared constants of the M2 experiments (in scratch they came from earlier cells)
K, T = 100.0, 30 / 365
params = HestonParams()

In [5]:
model_m3 = Policy(d_in = 3, m_out = 1)
out_m3 = train(model_m3, K, T, loss_fn=CVaRLoss(0.5), simulator=simulate_heston,
               sim_kwargs={"params": params}, use_var=True, v_scale=params.b,
               s_scale=(params.b * T) ** 0.5, n_steps=3000)

step: 0, loss: 3.6448347568511963
step: 100, loss: 2.6199333667755127
step: 200, loss: 2.5333921909332275
step: 300, loss: 2.4861369132995605
step: 400, loss: 2.5396032333374023
step: 500, loss: 2.5160369873046875
step: 600, loss: 2.4988937377929688
step: 700, loss: 2.614166259765625
step: 800, loss: 2.5377655029296875
step: 900, loss: 2.5619428157806396
step: 1000, loss: 2.472733497619629
step: 1100, loss: 2.5430612564086914
step: 1200, loss: 2.4954872131347656
step: 1300, loss: 2.492116928100586
step: 1400, loss: 2.557465076446533
step: 1500, loss: 2.541478157043457
step: 1600, loss: 2.5415778160095215
step: 1700, loss: 2.5586228370666504
step: 1800, loss: 2.526721954345703
step: 1900, loss: 2.4997267723083496
step: 2000, loss: 2.5385613441467285
step: 2100, loss: 2.5431623458862305
step: 2200, loss: 2.43837308883667
step: 2300, loss: 2.5452523231506348
step: 2400, loss: 2.529794692993164
step: 2500, loss: 2.514970302581787
step: 2600, loss: 2.5057380199432373
step: 2700, loss: 2.556

In [7]:
params = HestonParams()
with torch.no_grad():
    torch.manual_seed(0)
    test = simulate_heston(100_000, params=params)
    S_test, V_test = test["S"][:, :, 0], test["V"]

def eval_cvar(pol, alpha):
    with torch.no_grad():
        pnl = bm.hedge_pnl(S_test, pol, K, T, V=V_test)
    p0 = bm.summarize(pnl, alpha)["CVaR"]
    return {"p0": p0, **bm.summarize(pnl + p0, alpha)}

nn_pol = nn_policy(model_m3, K, T, use_var=True, s_scale=params.b)
bs_pol = bm.bs_delta_policy(K, 0.2)
bs_SV_pol = bm.bs_delta_stovol_policy(K)

result = {"NN": eval_cvar(nn_pol, 0.5),
          "BS_SV": eval_cvar(bs_SV_pol, 0.5),
          "BS": eval_cvar(bs_pol, 0.5)}

pd.DataFrame(result).T

,p0,mean,std,VaR,CVaR
NN,2.533553,0.836088,1.352719,-1.234363,2.192688e-07
BS_SV,2.670661,0.976190,1.349290,-1.455262,-2.822876e-07
BS,2.658308,0.961645,1.342169,-1.377534,-2.307129e-07
